# Classical ML Models — Guangyu

Working file for **Section 2** of `main.ipynb`: majority baseline, Logistic Regression, Decision Tree, Random Forest.

Every model runs over **4 cutoffs × 3 feature sets** with the same protocol: tune on validation → refit on train+val → test once.

## Plan & hints *(working notes — don't copy into `main.ipynb`)*

### Order of work — freeze Tue 14 Oct
1. 2.0 Config + `run()`; test LR / DT / RF at **cutoff 24 only**
2. Sanity-check numbers (below), then switch to all 4 cutoffs
3. 2.5 outputs: results CSV → tuning curves → predictions
4. Hand `results_classical.csv` to whoever builds Section 4

### Rules
- Use `a2_common` for split, preprocessing and metrics — never a custom split or scaler
- Tune on **val** only; refit on train+val; touch **test once** per cell
- Pass `SEED` to every estimator
- Each cutoff is a separate dataset — **never pool cutoffs**
- Training happens only in 2.1–2.4; 2.5 just saves, plots and predicts with already-fitted models

### Feature sets
| Name | Definition |
|---|---|
| `background` | `A2.feature_cols("baseline")` (14) |
| `behaviour` | `engineered` minus `baseline` (38) |
| `all` | `A2.feature_cols("engineered")` (52) |

### Models & grids (method: grid search, select on val macro-F1)
| Model | Bias / variance | Grid |
|---|---|---|
| Majority baseline | — | none, 4 cutoffs only |
| Logistic Regression | high bias, low variance | `C` ∈ {0.01, 0.03, 0.1, 0.3, 1, 3, 10} |
| Decision Tree | low bias, high variance | `max_depth` ∈ {2, 4, 6, 8, 10, 15, 20, None} |
| Random Forest | low bias, reduced variance (bagging) | `min_samples_leaf` ∈ {1, 5, 10, 20, 50} |

### `run()` contract
- For each cutoff × feature set: try grid → best on val → refit train+val → score test → append a row
- Keep train/val scores for **cutoff 24 + all** (for the tuning curves)
- Return the fitted best pipeline for **cutoff 24 + all** (for predictions)
- Agree the signature with Xinyan so the MLP plugs into the same loop

### Sanity checks
- Baseline test macro-F1 must match exactly: **0.345 / 0.352 / 0.371 / 0.385** (cutoffs 0 / 24 / 60 / 120)
- LR at cutoff 24: background ≈ 0.60, all ≈ 0.71
- ROC-AUC > 0.95 → suspect leakage before celebrating

### How the outputs answer the RQs
- **RQ1** (how early?): `all` row across cutoffs, lift over each cutoff's baseline, model vs model
- **RQ2** (behaviour vs background?): compare the 3 feature sets within each cutoff; gain = all − background; check it holds for every model
- Tuning curves → chosen-value explanation + bias–variance (theory–practice)
- Predictions → confusion matrix + specific cases (error analysis)

### Open items to confirm with the group
- My LR run replaces Wai Chun's `lr_*` / `rq1_*` CSVs (they used two different `C` grids)
- Add `behaviour` set to the shared definitions; rename feature set `baseline` → `background` in `main.ipynb`
- Model choice LR / DT / RF (fallback: swap RF for KNN if DT+RF is seen as too similar)
- README model-choice edits are still uncommitted

## 0. Setup

Imports, `SEED`, cutoffs, and the shared harness from `a2_common`.

## 2.0 Config

Feature sets (background / behaviour / all) and the model registry with each search grid.

## 2.1 Majority-class baseline

`DummyClassifier(strategy="most_frequent")`, no tuning. The reference every model must beat.

## 2.2 Logistic Regression

High bias, low variance. Tune `C` (inverse regularisation strength).

## 2.3 Decision Tree

Low bias, high variance. Tune `max_depth`.

## 2.4 Random Forest

Bagged trees: low bias, reduced variance. Tune `min_samples_leaf`.

## 2.5 Outputs

Results table, tuning-curve figure, and test predictions from the best model.

### 2.5.1 Results table — `results_classical.csv`

One row per model × cutoff × feature set: chosen parameter + test metrics. Feeds RQ1 and RQ2.

### 2.5.2 Tuning curves

Train vs validation macro-F1 against each hyperparameter (cutoff 24, all features), from scores kept in memory during tuning. Explains each chosen value.

### 2.5.3 Test predictions — `predictions_cutoff24.csv`

Predictions and probabilities from the best model, for the confusion matrix and error analysis.